In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyClassifier


In [2]:
dir_path = "data/"
train_transaction = pd.read_csv(f"{dir_path}train_transaction.csv")
train_identity = pd.read_csv(f"{dir_path}train_identity.csv")

In [3]:
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

In [4]:
del train_transaction, train_identity

In [5]:
train.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
train.shape

(590540, 434)

In [7]:
train.duplicated().sum()

np.int64(0)

#### Percentage of null values in every column

In [8]:
null_percent = (train.isnull().mean() * 100).round(2).sort_values(ascending=False)
print("Percentage of null values in every column\n")
print(null_percent.head(20))

Percentage of null values in every column

id_24    99.20
id_21    99.13
id_07    99.13
id_25    99.13
id_26    99.13
id_08    99.13
id_27    99.12
id_23    99.12
id_22    99.12
dist2    93.63
D7       93.41
id_18    92.36
D13      89.51
D14      89.47
D12      89.04
id_04    88.77
id_03    88.77
D6       87.61
id_33    87.59
id_09    87.31
dtype: float64


In [9]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), str(31)
memory usage: 1.9 GB


### Dataset quality and suitability\n
\\n
This dataset is suitable for credit-card fraud detection because it contains transaction-level records, a binary fraud target (`isFraud`), and transaction, card, address, and identity-related features. The merged training data contains 590,540 transactions and 434 columns. No exact duplicate rows were found.\n
\n
The dataset has important limitations: fraud is rare (approximately 3.5% of transactions), and many identity-related columns have substantial missingness. Missing values are therefore treated as potentially informative rather than automatically removing rows or columns. The class imbalance also means ROC-AUC should be complemented with a precision-recall metric.\n

In [10]:
train['TransactionAmt_to_mean_card1'] = train['TransactionAmt'] / train.groupby(['card1'])['TransactionAmt'].transform('mean')
train['TransactionAmt_to_mean_card4'] = train['TransactionAmt'] / train.groupby(['card4'])['TransactionAmt'].transform('mean')
train['TransactionAmt_to_std_card1'] = train['TransactionAmt'] / train.groupby(['card1'])['TransactionAmt'].transform('std')
train['TransactionAmt_to_std_card4'] = train['TransactionAmt'] / train.groupby(['card4'])['TransactionAmt'].transform('std')

train['id_02_to_mean_card1'] = train['id_02'] / train.groupby(['card1'])['id_02'].transform('mean')
train['id_02_to_mean_card4'] = train['id_02'] / train.groupby(['card4'])['id_02'].transform('mean')
train['id_02_to_std_card1'] = train['id_02'] / train.groupby(['card1'])['id_02'].transform('std')
train['id_02_to_std_card4'] = train['id_02'] / train.groupby(['card4'])['id_02'].transform('std')

train['D15_to_mean_card1'] = train['D15'] / train.groupby(['card1'])['D15'].transform('mean')
train['D15_to_mean_card4'] = train['D15'] / train.groupby(['card4'])['D15'].transform('mean')
train['D15_to_std_card1'] = train['D15'] / train.groupby(['card1'])['D15'].transform('std')
train['D15_to_std_card4'] = train['D15'] / train.groupby(['card4'])['D15'].transform('std')

train['D15_to_mean_addr1'] = train['D15'] / train.groupby(['addr1'])['D15'].transform('mean')
train['D15_to_mean_addr2'] = train['D15'] / train.groupby(['addr2'])['D15'].transform('mean')
train['D15_to_std_addr1'] = train['D15'] / train.groupby(['addr1'])['D15'].transform('std')
train['D15_to_std_addr2'] = train['D15'] / train.groupby(['addr2'])['D15'].transform('std')


In [11]:
# threshold = 0.90
# train = train.loc[:,train.isna().mean()<=threshold]

train = train.sort_values("TransactionDT").reset_index(drop=True)

dt = pd.to_datetime(train["TransactionDT"], unit="s", origin="2017-11-30")
months = (dt.dt.year - 2017) * 12 + dt.dt.month

month_summary = pd.DataFrame({
    "month": months,
    "isFraud": train["isFraud"]
}).groupby("month")["isFraud"].agg(rows="count", fraud_rate="mean")
print(month_summary, "\n")

TARGET = "isFraud"
DROP_COLS = ["TransactionID", "TransactionDT", TARGET]

# These group-statistic features were calculated using the full dataset before CV.
# Exclude them to prevent validation rows from influencing training features.
GROUP_STAT_FEATURES = [
    "TransactionAmt_to_mean_card1", "TransactionAmt_to_mean_card4",
    "TransactionAmt_to_std_card1", "TransactionAmt_to_std_card4",
    "id_02_to_mean_card1", "id_02_to_mean_card4",
    "id_02_to_std_card1", "id_02_to_std_card4",
    "D15_to_mean_card1", "D15_to_mean_card4",
    "D15_to_std_card1", "D15_to_std_card4",
    "D15_to_mean_addr1", "D15_to_mean_addr2",
    "D15_to_std_addr1", "D15_to_std_addr2",
]
feature_cols = [
    c for c in train.columns
    if c not in DROP_COLS and c not in GROUP_STAT_FEATURES
]
X = train[feature_cols].copy()
y = train[TARGET].values

         rows  fraud_rate
month                    
12     137321    0.025852
13      92585    0.040017
14      86021    0.040072
15     101632    0.039545
16      83655    0.033806
17      89326    0.034861 



In [12]:
# 1. Cast all string/object columns to 'category' and downcast floats/ints
cat_cols = X.select_dtypes(include=["object", "string"]).columns
X[cat_cols] = X[cat_cols].astype("category")

float_cols = X.select_dtypes(include=["float64"]).columns
X[float_cols] = X[float_cols].astype("float32")

int_cols = X.select_dtypes(include=["int64"]).columns
X[int_cols] = X[int_cols].astype("int32")

# 2. Verify zero unsupported dtypes remain (must print an empty list)
bad_cols = X.select_dtypes(exclude=["number", "bool", "category"]).columns.tolist()
print("Unsupported columns remaining:", bad_cols)



Unsupported columns remaining: []


### Leakage and validation strategy\n
\n
The full-dataset group-statistic features created above are excluded from the model because their values were calculated before the validation folds were formed. This avoids validation observations contributing to training features. The remaining preprocessing here is limited to dtype conversion and does not learn statistics from validation data.\n
\n
`GroupKFold` keeps each month entirely within either training or validation for a fold. However, it can train on later months while validating on an earlier month, so it measures month-group generalization rather than a strictly future-only deployment scenario.\n

In [13]:
gkf = GroupKFold(n_splits=6)

oof_preds = np.zeros(len(X))
fold_aucs = []
fold_aps = []
baseline_aucs = []
baseline_aps = []
importances = np.zeros(len(feature_cols))


In [14]:
lgb_params = dict(
    objective="binary",
    metric="auc",
    boosting_type="gbdt",
    n_estimators=2500,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    min_child_samples=50,
    random_state=42,
    n_jobs=-1,
)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=months), 1):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]
    val_month = months.iloc[val_idx].iloc[0]

    # Naive baseline: always predicts the training-fold fraud prevalence.
    baseline = DummyClassifier(strategy="prior")
    baseline.fit(np.zeros((len(y_tr), 1)), y_tr)
    baseline_preds = baseline.predict_proba(np.zeros((len(y_va), 1)))[:, 1]
    baseline_auc = roc_auc_score(y_va, baseline_preds)
    baseline_ap = average_precision_score(y_va, baseline_preds)
    baseline_aucs.append(baseline_auc)
    baseline_aps.append(baseline_ap)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    val_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[val_idx] = val_preds

    fold_auc = roc_auc_score(y_va, val_preds)
    fold_ap = average_precision_score(y_va, val_preds)
    fold_aucs.append(fold_auc)
    fold_aps.append(fold_ap)
    importances += model.booster_.feature_importance(importance_type="gain") / 6

    print(
        f"Fold {fold} (Val Month {val_month}) | "
        f"Best Iter: {model.best_iteration_:4d} | "
        f"ROC-AUC: {fold_auc:.5f} | PR-AUC/AP: {fold_ap:.5f} | "
        f"Baseline ROC-AUC: {baseline_auc:.5f} | Baseline AP: {baseline_ap:.5f}"
    )

oof_auc = roc_auc_score(y, oof_preds)
oof_ap = average_precision_score(y, oof_preds)

print(f"\nMean Fold ROC-AUC: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}")
print(f"Overall OOF ROC-AUC: {oof_auc:.5f}")
print(f"Mean Fold PR-AUC/AP: {np.mean(fold_aps):.5f} +/- {np.std(fold_aps):.5f}")
print(f"Overall OOF PR-AUC/AP: {oof_ap:.5f}")
print(f"Mean Baseline ROC-AUC: {np.mean(baseline_aucs):.5f}")
print(f"Mean Baseline PR-AUC/AP: {np.mean(baseline_aps):.5f}")


d:\tele\ai_pro\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 17113, number of negative: 436106
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.225019 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37516
[LightGBM] [Info] Number of data points in the train set: 453219, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.037759 -> initscore=-3.238047
[LightGBM] [Info] Start training from score -3.238047
Fold 1 (Val Month 12) | Best Iter:  796 | ROC-AUC: 0.91428 | PR-AUC/AP: 0.52361 | Baseline ROC-AUC: 0.50000 | Baseline AP: 0.02585


d:\tele\ai_pro\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16644, number of negative: 472264
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.252676 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39458
[LightGBM] [Info] Number of data points in the train set: 488908, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034043 -> initscore=-3.345488
[LightGBM] [Info] Start training from score -3.345488
Fold 2 (Val Month 15) | Best Iter:  433 | ROC-AUC: 0.92847 | PR-AUC/AP: 0.65023 | Baseline ROC-AUC: 0.50000 | Baseline AP: 0.03954


d:\tele\ai_pro\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16958, number of negative: 480997
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.412459 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39517
[LightGBM] [Info] Number of data points in the train set: 497955, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034055 -> initscore=-3.345121
[LightGBM] [Info] Start training from score -3.345121
Fold 3 (Val Month 13) | Best Iter:  545 | ROC-AUC: 0.92780 | PR-AUC/AP: 0.65572 | Baseline ROC-AUC: 0.50000 | Baseline AP: 0.04002


... (2 outputs left out)

d:\tele\ai_pro\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 17216, number of negative: 487303
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.307503 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39415
[LightGBM] [Info] Number of data points in the train set: 504519, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034124 -> initscore=-3.343047
[LightGBM] [Info] Start training from score -3.343047
Fold 5 (Val Month 14) | Best Iter:  509 | ROC-AUC: 0.93642 | PR-AUC/AP: 0.70242 | Baseline ROC-AUC: 0.50000 | Baseline AP: 0.04007


d:\tele\ai_pro\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 17835, number of negative: 489050
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.344993 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39411
[LightGBM] [Info] Number of data points in the train set: 506885, number of used features: 430
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035185 -> initscore=-3.311302
[LightGBM] [Info] Start training from score -3.311302
Fold 6 (Val Month 16) | Best Iter:  852 | ROC-AUC: 0.94138 | PR-AUC/AP: 0.66978 | Baseline ROC-AUC: 0.50000 | Baseline AP: 0.03381

Mean Fold ROC-AUC: 0.92880 +/- 0.00863
Overall OOF ROC-AUC: 0.92841
Mean Fo

### Evaluation and interpretation\n
\n
- **ROC-AUC** measures how well the model ranks fraudulent transactions above legitimate ones across thresholds. A value of 0.5 is random ranking; higher is better.\n
- **PR-AUC / Average Precision** focuses on precision and recall for the minority fraud class. Compare it with the fraud prevalence (the positive-class rate): a model should improve meaningfully over that reference.\n
- The `DummyClassifier(strategy='prior')` is a simple baseline that predicts the training-fold fraud rate for every transaction. The LightGBM model should outperform it, especially on PR-AUC.\n
- Because fraud is rare, ROC-AUC alone is insufficient to judge practical detection quality. A decision threshold and precision/recall trade-off can be examined later if needed.\n
\n
**Note:** Re-run the modified cells to generate the new baseline and PR-AUC results. The previously displayed ROC-AUC values were produced before the leakage-related features were excluded, so they should not be presented as results from the revised pipeline.\n